# Análisis de las decisiones de TPM (2010–2025): distribución, evolución y confiabilidad de un modelo predictivo

**Proyecto IMT2200 · Grupo 9 · Entrega 2 (video de avance)**

Este notebook responde tres preguntas con los datos que ya tenemos en `data/processed/`:

1. **¿Cómo se distribuyen las decisiones?** Barras con 103 mantenciones, 31 bajas y 27 subidas, y cómo cambian **año a año** (gráficos interactivos y animados).
2. **¿Por qué ese desbalance obliga a usar una línea base y métricas por clase?**
3. **¿Es confiable construir un modelo que anticipe la decisión de la próxima reunión de política monetaria (si la TPM subirá, bajará o se mantendrá)?** Se evalúa con una validación temporal (*walk-forward*) y se cuantifica cuánta confianza se puede tener en el resultado.

> **Unidad de análisis:** una fila por **reunión** de política monetaria (161 reuniones entre 2010 y 2025). El modelo, cuando lo construyamos, predecirá la decisión de la **próxima reunión**.

> **Gráficos dinámicos:** usamos **Plotly**, que genera gráficos interactivos dentro del notebook (hover con datos, animaciones, leyendas que se pueden activar y desactivar). Si no lo tienes instalado: `pip install plotly nbformat`. En cada explicación hay **un solo gráfico**.

## 0. Configuración

In [54]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_recall_fscore_support)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from IPython.display import display
except ImportError:          # fuera de Jupyter
    display = print

try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError as e:
    raise ImportError("Falta plotly. Instálalo con:  pip install plotly nbformat") from e

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"

# ---------- rutas (misma convención del notebook 01) ----------
NOTEBOOKS_DIR = Path.cwd()
ROOT_DIR = NOTEBOOKS_DIR.parent if NOTEBOOKS_DIR.name == "notebooks" else NOTEBOOKS_DIR
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

def ubicar(nombre: str) -> Path:
    """Busca un archivo en data/processed, data, la raíz o (último recurso) en todo el repo."""
    for base in (PROCESSED_DATA_DIR, ROOT_DIR / "data", ROOT_DIR, NOTEBOOKS_DIR):
        if (base / nombre).exists():
            return base / nombre
    hallados = list(ROOT_DIR.rglob(nombre))
    assert hallados, f"No encontré {nombre}. Revisa que esté en data/processed/."
    return hallados[0]

def leer_serie(nombre: str, col: str = None) -> pd.Series:
    d = pd.read_csv(ubicar(nombre), parse_dates=["fecha"]).set_index("fecha").sort_index()
    return d[col] if col else d.iloc[:, 0]

# ---------- estilo común ----------
ORDEN = ["baja", "mantiene", "sube"]
ETQ = {"baja": "Baja", "mantiene": "Mantiene", "sube": "Sube"}
COLOR = {"baja": "#0284c7", "mantiene": "#94a3b8", "sube": "#f59e0b"}   # azul / gris / ámbar
COLOR_E = {ETQ[k]: v for k, v in COLOR.items()}
ORDEN_E = [ETQ[k] for k in ORDEN]

print("Raíz del proyecto:", ROOT_DIR)

Raíz del proyecto: c:\Users\ferna\Desktop\tpm-chile


## 1. Carga del historial de decisiones y controles de calidad

Una fila por decisión de TPM (161 reuniones, 2010–2025). La etiqueta `decision` se construyó comparando la TPM del corte previo con la primera observación diaria posterior al anuncio.

In [55]:
R = (pd.read_csv(ubicar("reuniones_tpm_2010_2025_clean.csv"), parse_dates=["fecha_decision", "fecha_corte"])
       .sort_values("fecha_decision").reset_index(drop=True))
R["anio"] = R["fecha_decision"].dt.year

conteo = R["decision"].value_counts().reindex(ORDEN)

# ---- aserciones de control (mismo espíritu del ETL) ----
assert len(R) == 161, "Se esperaban 161 decisiones"
assert conteo.to_dict() == {"baja": 31, "mantiene": 103, "sube": 27}, conteo.to_dict()
assert R["fecha_decision"].is_unique
assert R["decision"].isin(ORDEN).all()
signo = np.sign(R["variacion_pb"]).map({-1.0: "baja", 0.0: "mantiene", 1.0: "sube"})
assert (signo == R["decision"]).all(), "La etiqueta no coincide con el signo de la variación en pb"

print(f"{len(R)} decisiones entre {R.fecha_decision.min():%d-%m-%Y} y {R.fecha_decision.max():%d-%m-%Y}")
print(conteo.to_string())

161 decisiones entre 14-01-2010 y 16-12-2025
decision
baja         31
mantiene    103
sube         27


## 2. Distribución de decisiones: ¿qué tan desbalanceado está el problema?

Dale *play* o arrastra el *slider* de años: las barras muestran el **acumulado** de decisiones hasta el año seleccionado. En 2025 queda el total: **103 mantenciones, 31 bajas y 27 subidas**.

In [56]:
por_anio = (R.groupby(["anio", "decision"]).size().unstack(fill_value=0)
              .reindex(columns=ORDEN, fill_value=0))
acum = por_anio.cumsum()

acum_long = (acum.reset_index().melt(id_vars="anio", var_name="clave", value_name="acumulado"))
acum_long["Decisión"] = acum_long["clave"].map(ETQ)

fig = px.bar(acum_long, x="Decisión", y="acumulado", color="Decisión", text="acumulado",
             animation_frame="anio", color_discrete_map=COLOR_E,
             category_orders={"Decisión": ORDEN_E}, range_y=[0, 120],
             title="Decisiones de TPM acumuladas hasta el año seleccionado")
fig.update_traces(textposition="outside", textfont_size=15)
fig.update_layout(showlegend=False, yaxis_title="Reuniones acumuladas", xaxis_title="", height=500)
# velocidad de la animación (ms por cuadro y por transición)
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 700
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300
fig.show()

**Lectura:** casi dos de cada tres reuniones terminan en *mantiene* (64,0 %). Las bajas (19,3 %) y las subidas (16,8 %) son minorías. Es un problema **desbalanceado**: las clases que más interesa anticipar son justamente las menos frecuentes.

## 3. Evolución por año: mantenciones, subidas y bajas

Cada barra es un año; sus colores muestran cuántas reuniones terminaron en cada decisión. Hasta 2017 hubo 12 reuniones al año y desde 2018, 8 (2020 tuvo 9 por una reunión extraordinaria), por eso las barras de los primeros años son más altas.

In [57]:
anios = por_anio.index.tolist()
fig = go.Figure()
for k in ORDEN:
    fig.add_trace(go.Bar(x=anios, y=por_anio[k], name=ETQ[k], marker_color=COLOR[k],
                         hovertemplate="%{x}: %{y} reuniones<extra>" + ETQ[k] + "</extra>"))
fig.update_layout(barmode="stack", title="Decisiones de TPM por año (número de reuniones)", height=500,
                  xaxis=dict(dtick=1, title=""), yaxis_title="Reuniones", hovermode="x unified")
fig.show()

**Lectura:** la mezcla de decisiones **cambia con el ciclo económico**. Hubo subidas en 2010–11 (7 y 5 de 12 reuniones) y en 2022 (7 de 8); bajas en 2014 (6), 2017 (4) y 2023–24 (4 y 7 de 8); y 2016 tuvo las 12 reuniones en mantención. Esto importa para el modelo: un período no se parece al siguiente.

## 4. Por qué el desbalance obliga a usar una línea base y métricas por clase

Si un "modelo" respondiera **siempre "mantiene"**, ¿qué tan bien se vería? Lo calculamos sobre todo el historial. *(Es un ejercicio descriptivo para ilustrar el problema, no una evaluación fuera de muestra; la evaluación honesta viene en la sección 5.)*

In [58]:
y_all = R["decision"].values
pred_trivial = np.full(len(y_all), "mantiene")

pr, rc, f1c, _ = precision_recall_fscore_support(y_all, pred_trivial, labels=ORDEN, zero_division=0)
por_clase = pd.DataFrame({"precision": pr, "recall": rc, "F1": f1c}, index=[ETQ[k] for k in ORDEN]).round(3)
display(por_clase)

p_clases = conteo / conteo.sum()
resumen = pd.DataFrame({
    "Métrica": ["Accuracy", "Balanced accuracy", "F1 macro"],
    "Siempre 'mantiene'": [accuracy_score(y_all, pred_trivial),
                           balanced_accuracy_score(y_all, pred_trivial),
                           f1_score(y_all, pred_trivial, average="macro", labels=ORDEN, zero_division=0)]})
resumen["Siempre 'mantiene'"] = resumen["Siempre 'mantiene'"].round(3)
print(f"Accuracy esperado de adivinar al azar según las proporciones: {(p_clases**2).sum():.3f}")

fig = px.bar(resumen, x="Métrica", y="Siempre 'mantiene'", text="Siempre 'mantiene'",
             color="Métrica", color_discrete_sequence=["#64748b", "#f59e0b", "#0284c7"],
             title="Un modelo trivial ('siempre mantiene') engaña si solo miramos el accuracy")
fig.update_traces(textposition="outside", textfont_size=15)
fig.update_layout(showlegend=False, yaxis_range=[0, 1.05], yaxis_title="Valor de la métrica", xaxis_title="", height=420)
fig.show()

,precision,recall,F1
Baja,0.00,0.0,0.00
Mantiene,0.64,1.0,0.78
Sube,0.00,0.0,0.00


Accuracy esperado de adivinar al azar según las proporciones: 0.474


**Qué muestra esto (y qué justifica el resto del análisis):**

- Un modelo que no aprende nada logra ~64 % de *accuracy*. Por eso el *accuracy* solo **no sirve** para juzgar un modelo: **necesitamos una línea base** que fije el piso a superar.
- Ese mismo modelo tiene *recall* **0 %** en bajas y en subidas: nunca detecta un movimiento, que es lo que realmente importa. Por eso necesitamos **métricas por clase** (precision, recall y F1 de cada decisión) y una medida equilibrada (*balanced accuracy* o *F1 macro*).
- La línea base no puede ser solo "siempre mantiene": hay otra mucho más exigente, **"repetir la decisión anterior"** (persistencia), porque los ciclos de TPM duran varias reuniones. La evaluamos en la sección 5.

## 5. ¿Es confiable construir un modelo que anticipe la decisión de la próxima reunión?

Para responder no basta con entrenar un modelo y mirar un número. Revisamos, en este orden, **seis condiciones** que determinan cuánta confianza merece cualquier resultado: (1) cuántos datos hay, (2) si el pasado ayuda a predecir el futuro, (3) si el problema es estable en el tiempo, (4) si las variables estaban realmente disponibles antes de cada reunión, (5) si el modelo supera a una línea base exigente y (6) cuánta incertidumbre tiene esa comparación.

### 5.1 Tamaño de muestra y poder estadístico

Con 161 decisiones y solo 31 y 27 en las clases minoritarias, el modelo parte con muy poca información sobre bajas y subidas. Usaremos una evaluación **walk-forward** (ventana creciente): entrenar con las primeras *k* decisiones y predecir la siguiente, repitiendo desde la decisión 61 en adelante. Así cada predicción solo usa información previa.

In [59]:
N0 = 60                     # tamaño mínimo de entrenamiento (5 años de reuniones)
Y = R["decision"].values
IDX = np.arange(N0, len(R))
Y_TEST = Y[IDX]

ent_ini = pd.Series(Y[:N0]).value_counts().reindex(ORDEN)
ent_fin = pd.Series(Y[:-1]).value_counts().reindex(ORDEN)
test_c = pd.Series(Y_TEST).value_counts().reindex(ORDEN)

poder = pd.DataFrame({"Entrenamiento inicial (60)": ent_ini, "Entrenamiento final (160)": ent_fin,
                      f"Evaluación ({len(IDX)} reuniones)": test_c})
poder.index = [ETQ[k] for k in ORDEN]
display(poder)

# Regla empírica: ~10 eventos por variable (EPV). Es una referencia, no una ley.
print(f"Variables que 'soporta' la clase más pequeña: ~{ent_ini.min()/10:.1f} al inicio y ~{ent_fin.min()/10:.1f} al final (regla de 10 eventos por variable)")

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

print("\nSi un modelo acertara 3 de cada 4 subidas en la evaluación:")
lo, hi = wilson(round(0.75 * test_c["sube"]), test_c["sube"])
print(f"  recall observado 75% con n={test_c['sube']} subidas -> IC95% [{lo:.0%}, {hi:.0%}]")

,Entrenamiento inicial (60),Entrenamiento final (160),Evaluación (101 reuniones)
Baja,9,30,22
Mantiene,39,103,64
Sube,12,27,15


Variables que 'soporta' la clase más pequeña: ~0.9 al inicio y ~2.7 al final (regla de 10 eventos por variable)

Si un modelo acertara 3 de cada 4 subidas en la evaluación:
  recall observado 75% con n=15 subidas -> IC95% [48%, 89%]


**Conclusión 5.1:** al comienzo hay solo ~9 bajas y ~12 subidas para aprender, y la evaluación contiene solo 22 bajas y 15 subidas. Cada acierto o error de más mueve mucho el *recall* de la clase. Se pueden **usar** pocas variables y los resultados por clase tendrán **intervalos de confianza anchos**.

### 5.2 ¿El pasado ayuda a predecir el futuro? Dependencia temporal

Si las decisiones fuesen independientes entre reuniones, repetir la decisión anterior acertaría solo cuando la casualidad lo permitiera. Medimos cuánto se repite realmente y lo comparamos con barajar el orden de las decisiones (test de permutación).

In [60]:
dec = R["decision"].values
trans = (pd.crosstab(pd.Series(dec[:-1], name="actual"), pd.Series(dec[1:], name="siguiente"))
           .reindex(index=ORDEN, columns=ORDEN, fill_value=0))
trans_pct = trans.div(trans.sum(axis=1), axis=0) * 100

rng = np.random.default_rng(42)
obs_repite = (dec[1:] == dec[:-1]).mean()
perm = np.array([(lambda s: (s[1:] == s[:-1]).mean())(rng.permutation(dec)) for _ in range(5000)])
p_perm = (perm >= obs_repite).mean()
chi = stats.chi2_contingency(trans.values)

# después de un movimiento (sube/baja): ¿qué viene?
tras_mov = [(a, b) for a, b in zip(dec[:-1], dec[1:]) if a != "mantiene"]
pct_mismo = np.mean([a == b for a, b in tras_mov]); mant = np.mean([b == "mantiene" for a, b in tras_mov])
inversion = np.mean([(b != "mantiene") and (a != b) for a, b in tras_mov])

print(f"La decisión se repite en {obs_repite:.1%} de las reuniones (con orden aleatorio se esperaría {perm.mean():.1%}); p-valor de permutación {'< 0,001' if p_perm == 0 else '= ' + format(p_perm, '.4f')}")
print(f"Chi² de independencia de la matriz de transición: {chi.statistic:.1f} (p = {chi.pvalue:.1e})")
print(f"Tras una subida o baja: mismo sentido {pct_mismo:.0%}, vuelve a mantener {mant:.0%}, cambio inmediato de sentido {inversion:.0%} (n={len(tras_mov)})")

fig = px.imshow(trans_pct.round(1).rename(index=ETQ, columns=ETQ), text_auto=".0f", aspect="auto",
                color_continuous_scale="Blues", zmin=0, zmax=100,
                labels=dict(x="Decisión siguiente", y="Decisión actual", color="%"),
                title="Matriz de transición: ¿qué decisión sigue a cuál? (% por fila)")
fig.update_layout(height=420)
fig.show()


La decisión se repite en 75.6% de las reuniones (con orden aleatorio se esperaría 47.1%); p-valor de permutación < 0,001
Chi² de independencia de la matriz de transición: 110.1 (p = 6.9e-23)
Tras una subida o baja: mismo sentido 67%, vuelve a mantener 33%, cambio inmediato de sentido 0% (n=57)


**Conclusión 5.2:** hay **fuerte persistencia**: las decisiones forman rachas (ciclos de alzas, de recortes o de pausa): se repiten en 75,6 % de las reuniones frente a 47,1 % si el orden fuera aleatorio, y en este historial **nunca** se pasó de subir a bajar (ni al revés) de una reunión a la siguiente (0 de 57 movimientos). Esto es una buena noticia (hay estructura) y una mala (la línea base de "repetir lo anterior" será **difícil de superar**, y un modelo que solo reproduce esa persistencia no aporta nada nuevo).

### 5.3 ¿Es estable el problema en el tiempo? (no estacionariedad)

In [61]:
periodo = pd.cut(R["anio"], [2009, 2013, 2017, 2021, 2025], labels=["2010–13", "2014–17", "2018–21", "2022–25"])
ct = pd.crosstab(periodo, R["decision"]).reindex(columns=ORDEN)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
chi2_per = stats.chi2_contingency(ct.values)
cramer = np.sqrt(chi2_per.statistic / (ct.values.sum() * (min(ct.shape) - 1)))

tabla_per = ct_pct.round(0).astype(int).astype(str) + "%"
tabla_per.columns = [ETQ[k] for k in ORDEN]
tabla_per.insert(0, "Reuniones", ct.sum(axis=1))
tabla_per.index.name = "Período"
display(tabla_per)
print(f"Chi² período × decisión = {chi2_per.statistic:.1f}, p = {chi2_per.pvalue:.4f}, V de Cramér = {cramer:.2f}")

,Reuniones,Baja,Mantiene,Sube
Período,,,,
2010–13,48,6%,69%,25%
2014–17,48,21%,75%,4%
2018–21,33,15%,67%,18%
2022–25,32,41%,38%,22%


Chi² período × decisión = 23.8, p = 0.0006, V de Cramér = 0.27


**Conclusión 5.3:** la distribución **no es la misma en todos los períodos** (diferencia estadísticamente significativa). En 2010–13 hubo 25 % de subidas y solo 6 % de bajas; en 2014–17 casi no hubo subidas (4 %); y en 2022–25 las mantenciones caen a 38 % mientras las bajas suben a 41 %. Un modelo entrenado en un régimen puede no servir en el siguiente, por eso la validación debe ser **cronológica** (nunca aleatoria) y los resultados dependerán del período evaluado.

### 5.4 Variables disponibles *antes* de cada reunión (sin fuga de información)

Cada predictor se mide **al cierre del día hábil anterior** a la reunión, usando solo lo que ya estaba publicado. Los rezagos se aplican así:

| Variable | Qué se usa | Regla de disponibilidad (conservadora) |
|---|---|---|
| TPM previa | Nivel al corte | Dato del corte |
| Decisiones previas | Variación en pb de la reunión anterior y suma de las 3 últimas | Ya conocidas |
| IPC (12 m y mensual) | Inflación acumulada 12 meses y variación del mes | Mes anterior si el corte es el día ≥ 9; si no, dos meses atrás (el INE publica ~el día 8) |
| Imacec | Variación anual | Dos meses atrás si el corte es el día ≥ 3; si no, tres |
| Desempleo | Nivel y cambio en 3 meses | Tres meses atrás (publicación a fin de mes, trimestre móvil) |
| Dólar y cobre | Variación de las últimas 21 ruedas | Último dato ≤ corte |
| **EEE** (expectativa de TPM) | Brecha entre la expectativa mediana y la TPM actual | La EEE se publica ~el día 10 de cada mes (calendario BCCh: entre el 10 y el 13). Se usa la del mes si el corte es el día ≥ 13; si no, la del mes anterior |

> **Limitaciones declaradas:** no hay versiones históricas de los datos (Imacec y otras series se revisan), y las reglas de rezago son aproximaciones que el grupo debe contrastar con calendarios de publicación independientes. Además, la serie EEE conviene revisarla en su definición exacta (horizonte de la pregunta de TPM) antes de la versión final.

In [62]:
ipc = leer_serie("ipc_general_mensual_clean.csv")
imacec = leer_serie("imacec_empalmado_original_clean.csv")
desemp = leer_serie("desempleo_total_mensual_clean.csv")
dolar = leer_serie("clp_usd_clean.csv")
cobre = leer_serie("cobre_cochilco_diario_2010_2025.csv", "cu_usd_lb")
eee = leer_serie("exp_tpm_mensual_mediana_clean.csv")

ipc_12m = ((1 + ipc / 100).rolling(12).apply(np.prod, raw=True) - 1) * 100   # inflación acumulada 12 meses
imacec_yoy = imacec.pct_change(12) * 100
desemp_d3 = desemp.diff(3)

def valor_mensual(serie, mes):
    return serie.get(mes, np.nan)

def var_diaria(serie, corte, ruedas=21):
    s = serie[:corte].dropna()
    return (s.iloc[-1] / s.iloc[-1 - ruedas] - 1) * 100 if len(s) > ruedas else np.nan

filas = []
for _, r in R.iterrows():
    corte = r["fecha_corte"]
    mes = corte.to_period("M").to_timestamp()
    atras = lambda k: mes - pd.DateOffset(months=k)
    lag_ipc = 1 if corte.day >= 9 else 2
    lag_ima = 2 if corte.day >= 3 else 3
    mes_eee = mes if corte.day >= 13 else atras(1)
    filas.append(dict(
        tpm_previa=r["tpm_previa_pct"],
        ipc_12m=valor_mensual(ipc_12m, atras(lag_ipc)), ipc_mom=valor_mensual(ipc, atras(lag_ipc)),
        imacec_yoy=valor_mensual(imacec_yoy, atras(lag_ima)),
        desempleo=valor_mensual(desemp, atras(3)), desempleo_d3=valor_mensual(desemp_d3, atras(3)),
        dolar_1m=var_diaria(dolar, corte), cobre_1m=var_diaria(cobre, corte),
        exp_gap_pub=valor_mensual(eee, mes_eee) - r["tpm_previa_pct"],      # EEE respetando fecha de publicación
        exp_gap_fuga=valor_mensual(eee, mes) - r["tpm_previa_pct"],         # EEE del mismo mes SIN respetarla (solo para contraste)
        eee_mismo_mes=int(corte.day >= 13)))
F = pd.DataFrame(filas)

pb = R["variacion_pb"]
F["prev_pb"] = pb.shift(1).fillna(0).values
F["net3"] = pb.shift(1).rolling(3).sum().fillna(0).values

ESTADO = ["tpm_previa", "prev_pb", "net3"]
MACRO = ["ipc_12m", "ipc_mom", "imacec_yoy", "desempleo", "desempleo_d3", "dolar_1m", "cobre_1m"]
assert len(F) == len(R)
print("Nulos por variable (se imputan con la mediana del entrenamiento):")
print(F[ESTADO + MACRO + ["exp_gap_pub"]].isna().sum()[lambda s: s > 0].to_string())

Nulos por variable (se imputan con la mediana del entrenamiento):
desempleo       5
desempleo_d3    8
cobre_1m        1


### 5.5 Evaluación walk-forward: líneas base frente a modelos

Se comparan **tres líneas base** y **dos tipos de modelo** (regresión logística multiclase, que es la prevista en el proyecto, y un árbol de decisión poco profundo) sobre **seis conjuntos de variables**. Todo se reentrena en cada reunión usando solo el pasado; los valores faltantes se imputan con la mediana del entrenamiento.

- **Líneas base:** (a) *Siempre la clase mayoritaria del entrenamiento*, (b) *Repetir la decisión anterior* (persistencia), (c) *Azar según las proporciones del entrenamiento* (promedio de 200 simulaciones).
- **Conjuntos de variables:** solo macro · solo estado (TPM previa y decisiones previas) · estado + macro · solo EEE publicada · estado + EEE publicada · ⚠ EEE del mismo mes sin respetar su publicación (caso de **fuga**, para medir cuánto distorsionaría).

In [63]:
CLASES = ORDEN

def metricas(y, p):
    pr, rc, f, _ = precision_recall_fscore_support(y, p, labels=CLASES, zero_division=0)
    m = dict(accuracy=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
             f1_macro=f1_score(y, p, average="macro", labels=CLASES, zero_division=0),
             mcc=matthews_corrcoef(y, p))
    for c, a, b, d in zip(CLASES, pr, rc, f):
        m[f"P_{c}"], m[f"R_{c}"], m[f"F1_{c}"] = a, b, d
    return m

def f1m(y, p):
    return f1_score(y, p, average="macro", labels=CLASES, zero_division=0)

def lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
def arbol():
    return make_pipeline(SimpleImputer(strategy="median"),
                         DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, class_weight="balanced", random_state=0))

def walk_forward(fabrica, cols):
    pred = []
    for i in IDX:
        m = fabrica()
        m.fit(F.loc[:i - 1, cols], Y[:i])          # solo reuniones anteriores a la i
        pred.append(m.predict(F.loc[[i], cols])[0])
    return np.array(pred)

# ---------- líneas base ----------
PRED = {}
PRED["Base: siempre mayoritaria"] = np.array([pd.Series(Y[:i]).value_counts().idxmax() for i in IDX])
PRED["Base: repetir decisión anterior"] = np.array([Y[i - 1] for i in IDX])

priors = [pd.Series(Y[:i]).value_counts(normalize=True).reindex(CLASES, fill_value=0).values for i in IDX]
azar = []
for s in range(200):
    g = np.random.default_rng(s)
    azar.append(metricas(Y_TEST, np.array([g.choice(CLASES, p=p) for p in priors])))
azar_m = pd.DataFrame(azar).mean().to_dict()

# ---------- modelos ----------
CONJUNTOS = {
    "Solo macro":                        MACRO,
    "Solo estado":                       ESTADO,
    "Estado + macro":                    ESTADO + MACRO,
    "Solo EEE publicada":                ["exp_gap_pub"],
    "Estado + EEE publicada":            ESTADO + ["exp_gap_pub"],
    "⚠ EEE mismo mes (fuga)":            ["exp_gap_fuga"],
}
for nombre, cols in CONJUNTOS.items():
    PRED[f"LR | {nombre}"] = walk_forward(lr, cols)
    PRED[f"Árbol | {nombre}"] = walk_forward(arbol, cols)

RES = pd.DataFrame({k: metricas(Y_TEST, v) for k, v in PRED.items()}).T
RES.loc["Base: azar según proporciones"] = pd.Series(azar_m)
RES = RES.astype(float)

columnas_resumen = ["accuracy", "bal_acc", "f1_macro", "mcc", "F1_baja", "F1_mantiene", "F1_sube", "R_baja", "R_sube"]
display(RES[columnas_resumen].sort_values("f1_macro", ascending=False).round(3))

,accuracy,bal_acc,f1_macro,mcc,F1_baja,F1_mantiene,F1_sube,R_baja,R_sube
Árbol | ⚠ EEE mismo mes (fuga),0.891,0.835,0.856,0.789,0.857,0.924,0.786,0.818,0.733
LR | ⚠ EEE mismo mes (fuga),0.891,0.835,0.856,0.789,0.857,0.924,0.786,0.818,0.733
LR | Solo EEE publicada,0.871,0.815,0.838,0.750,0.791,0.909,0.815,0.773,0.733
Árbol | Solo EEE publicada,0.871,0.815,0.838,0.750,0.791,0.909,0.815,0.773,0.733
Árbol | Estado + EEE publicada,0.792,0.766,0.757,0.617,0.750,0.832,0.690,0.818,0.667
LR | Estado + EEE publicada,0.782,0.788,0.757,0.620,0.745,0.817,0.710,0.864,0.733
Base: repetir decisión anterior,0.733,0.685,0.687,0.491,0.605,0.791,0.667,0.591,0.667
Árbol | Solo estado,0.683,0.689,0.655,0.459,0.604,0.735,0.625,0.727,0.667
LR | Estado + macro,0.604,0.668,0.603,0.371,0.464,0.642,0.703,0.591,0.867
LR | Solo estado,0.624,0.654,0.602,0.375,0.560,0.667,0.579,0.636,0.733


In [64]:
# F1 por clase de las líneas base y la regresión logística (clic en la leyenda para ocultar/mostrar clases)
orden_modelos = [m for m in RES.sort_values("f1_macro").index if not m.startswith("Árbol")]   # el árbol está en la tabla
fig = go.Figure()
for k in ORDEN:
    fig.add_trace(go.Bar(y=orden_modelos, x=RES.loc[orden_modelos, f"F1_{k}"], name=f"F1 {ETQ[k]}",
                         orientation="h", marker_color=COLOR[k],
                         hovertemplate="%{y}<br>F1 " + ETQ[k] + ": %{x:.2f}<extra></extra>"))
fig.add_trace(go.Scatter(y=orden_modelos, x=RES.loc[orden_modelos, "f1_macro"], mode="markers", name="F1 macro",
                         marker=dict(color="#111827", symbol="diamond", size=11),
                         hovertemplate="%{y}<br>F1 macro: %{x:.2f}<extra></extra>"))
fig.update_layout(barmode="group", title="F1 por clase y F1 macro: líneas base y regresión logística (walk-forward)",
                  xaxis=dict(title="F1", range=[0, 1]), height=620, legend=dict(orientation="h", y=1.04))
fig.show()

**Lectura de 5.5** (evaluación sobre 101 reuniones: 64 mantienes, 22 bajas y 15 subidas):

- **Piso a superar:** "siempre mayoritaria" tiene accuracy 0,63 pero F1 macro 0,26; el azar según proporciones, F1 macro ≈ 0,32. La línea base realmente exigente es **repetir la decisión anterior**: accuracy 0,73 y F1 macro 0,69.
- **Solo indicadores macro (IPC, Imacec, desempleo, dólar, cobre):** F1 macro 0,40 con regresión logística y 0,47 con árbol; es decir, **cerca del azar y muy por debajo de la persistencia**.
- **Agregar macro al estado no ayuda:** con regresión logística el F1 macro queda igual (0,60) y con árbol cae de 0,66 a 0,47. Con tan pocos eventos por clase, más variables significan sobreajuste.
- **Solo la brecha de la EEE** (expectativa de TPM menos TPM actual) alcanza accuracy 0,87 y F1 macro 0,84, con buen recall en ambas clases minoritarias. Combinarla con otras variables **empeora** el resultado (0,76).

### 5.6 ¿Cuánta incertidumbre tiene la comparación?

Con solo 101 reuniones evaluadas, una diferencia de F1 puede ser ruido. Cuantificamos con un **bootstrap por bloques** (bloques de 6 reuniones, que preservan las rachas) tanto el F1 macro de cada enfoque como la **diferencia frente a la línea base de persistencia**.

In [65]:
def bloques(n, L, g):
    nb = int(np.ceil(n / L))
    ini = g.integers(0, n - L + 1, nb)
    return np.concatenate([np.arange(s, s + L) for s in ini])[:n]

g = np.random.default_rng(42)
B, L = 1500, 6
candidatos = ["Base: repetir decisión anterior", "Base: siempre mayoritaria", "LR | Solo macro",
              "LR | Solo estado", "LR | Estado + macro", "LR | Estado + EEE publicada", "LR | Solo EEE publicada"]
boot = {k: [] for k in candidatos}
n = len(Y_TEST)
for _ in range(B):
    ii = bloques(n, L, g)
    for k in candidatos:
        boot[k].append(f1m(Y_TEST[ii], PRED[k][ii]))

base_k = "Base: repetir decisión anterior"
filas_ic = []
for k in candidatos:
    a = np.array(boot[k]); d = a - np.array(boot[base_k])
    filas_ic.append(dict(enfoque=k, f1_macro=f1m(Y_TEST, PRED[k]),
                         ic_inf=np.percentile(a, 2.5), ic_sup=np.percentile(a, 97.5),
                         delta_vs_persistencia=f1m(Y_TEST, PRED[k]) - f1m(Y_TEST, PRED[base_k]),
                         delta_inf=np.percentile(d, 2.5), delta_sup=np.percentile(d, 97.5),
                         prob_mejor=(d > 0).mean()))
IC = pd.DataFrame(filas_ic).set_index("enfoque")
display(IC.round(3))

fig = go.Figure(go.Scatter(
    y=IC.index, x=IC["f1_macro"], mode="markers",
    error_x=dict(type="data", symmetric=False, array=IC["ic_sup"] - IC["f1_macro"], arrayminus=IC["f1_macro"] - IC["ic_inf"]),
    marker=dict(size=11, color="#0284c7"),
    hovertemplate="%{y}<br>F1 macro %{x:.2f}<extra></extra>"))
fig.update_layout(title="F1 macro con intervalo de confianza 95% (bootstrap por bloques)",
                  xaxis=dict(title="F1 macro", range=[0, 1]), height=450, yaxis=dict(autorange="reversed"))
fig.show()

,f1_macro,ic_inf,ic_sup,delta_vs_persistencia,delta_inf,delta_sup,prob_mejor
enfoque,,,,,,,
Base: repetir decisión anterior,0.687,0.450,0.821,0.000,0.000,0.000,0.000
Base: siempre mayoritaria,0.259,0.215,0.293,-0.429,-0.584,-0.172,0.000
LR | Solo macro,0.395,0.263,0.537,-0.292,-0.458,-0.063,0.004
LR | Solo estado,0.602,0.387,0.752,-0.085,-0.199,0.007,0.036
LR | Estado + macro,0.603,0.427,0.751,-0.084,-0.220,0.097,0.181
LR | Estado + EEE publicada,0.757,0.562,0.850,0.070,-0.027,0.164,0.916
LR | Solo EEE publicada,0.838,0.605,0.924,0.151,0.032,0.277,0.991


**Lectura de 5.6:** los intervalos son **anchos**: incluso la persistencia tiene un IC95 % de F1 macro entre 0,45 y 0,82. Con esta muestra solo se pueden distinguir diferencias grandes:

- Los modelos con macro quedan **claramente por debajo** de la persistencia o, en el mejor caso, indistinguibles de ella (Estado + macro: Δ = −0,08, IC [−0,22; +0,10]).
- Solo la **EEE publicada** supera a la persistencia con respaldo estadístico (Δ = +0,15, IC95 % [+0,03; +0,28]; probabilidad de ser mejor ≈ 99 %), aunque su propio F1 macro tiene un IC amplio ([0,61; 0,92]).

### 5.7 Sensibilidad: ¿qué pasa con la EEE y con el momento de su publicación?

La EEE (expectativa de TPM de analistas) es lo único que se acerca a la señal. Revisamos dos cosas: (1) cuánto cambiaría el resultado si **no** se respetara su fecha de publicación (fuga), y (2) si el resultado se sostiene en reuniones donde la EEE **del mismo mes** aún no estaba publicada.

In [66]:
fuga = f1m(Y_TEST, PRED["LR | ⚠ EEE mismo mes (fuga)"])
legit = f1m(Y_TEST, PRED["LR | Solo EEE publicada"])
print(f"F1 macro con EEE respetando publicación: {legit:.3f}   |   sin respetarla (fuga): {fuga:.3f}   |   diferencia: {fuga - legit:+.3f}")

mismo = F.loc[IDX, "eee_mismo_mes"].values == 1
filas_s = []
for etiqueta, mascara in [("EEE del mes ya publicada (corte día ≥ 13)", mismo), ("EEE del mes aún no publicada (usa la del mes anterior)", ~mismo)]:
    for k in ["Base: repetir decisión anterior", "LR | Solo EEE publicada"]:
        filas_s.append(dict(subconjunto=etiqueta, enfoque=k, reuniones=int(mascara.sum()),
                            accuracy=accuracy_score(Y_TEST[mascara], PRED[k][mascara]),
                            f1_macro=f1m(Y_TEST[mascara], PRED[k][mascara])))
sens = pd.DataFrame(filas_s)
display(sens.round(3))

fig = px.bar(sens, x="subconjunto", y="f1_macro", color="enfoque", barmode="group", text=sens["f1_macro"].round(2),
             color_discrete_sequence=["#94a3b8", "#0284c7"],
             title="F1 macro según si la EEE del mes ya estaba publicada")
fig.update_layout(yaxis=dict(range=[0, 1], title="F1 macro"), xaxis_title="", height=430)
fig.show()

F1 macro con EEE respetando publicación: 0.838   |   sin respetarla (fuga): 0.856   |   diferencia: +0.017


,subconjunto,enfoque,reuniones,accuracy,f1_macro
0,EEE del mes ya publicada (corte día ≥ 13),Base: repetir decisión anterior,66,0.727,0.656
1,EEE del mes ya publicada (corte día ≥ 13),LR | Solo EEE publicada,66,0.864,0.804
2,EEE del mes aún no publicada (usa la del mes anterior),Base: repetir decisión anterior,35,0.743,0.697
3,EEE del mes aún no publicada (usa la del mes anterior),LR | Solo EEE publicada,35,0.886,0.860


**Lectura de 5.7:**

- Ignorar la fecha de publicación (usar siempre la EEE del mismo mes) subiría el F1 macro solo de 0,84 a 0,86, o sea, **la fuga no explica el resultado**.
- El modelo con EEE rinde bien incluso en las 35 reuniones de inicio de mes, donde la EEE del mes aún no estaba publicada y se usó la anterior (F1 macro 0,86 vs. 0,80 en las otras 66). Esto sugiere que la serie recoge expectativas con horizonte de aproximadamente un mes, pero **no podemos confirmar su horizonte exacto con estos datos**: hay que revisar en la fuente a qué pregunta de la encuesta corresponde la serie.
- Aun siendo legítima, la EEE es el **pronóstico de analistas del propio mercado**: usarla equivale a "copiar el consenso", no a anticipar con indicadores macro independientes.

### 5.8 Tablero de confiabilidad

Resumen de las seis condiciones con la evidencia calculada arriba (🟢 favorable · 🟡 con reservas · 🔴 desfavorable).

In [67]:
f1_pers = f1m(Y_TEST, PRED["Base: repetir decisión anterior"])
f1_macro_solo = f1m(Y_TEST, PRED["LR | Solo macro"])
f1_estmac = f1m(Y_TEST, PRED["LR | Estado + macro"])
f1_eee = f1m(Y_TEST, PRED["LR | Solo EEE publicada"])
d_eee = IC.loc["LR | Solo EEE publicada"]
d_mac = IC.loc["LR | Estado + macro"]
anch_pers = IC.loc[base_k, "ic_sup"] - IC.loc[base_k, "ic_inf"]

tablero = pd.DataFrame([
    ("1. Tamaño de muestra", "🔴", f"161 decisiones; al inicio solo {ent_ini['baja']} bajas y {ent_ini['sube']} subidas "
        f"(~{ent_ini.min()/10:.1f} variable por la regla de 10 eventos)."),
    ("2. Dependencia temporal", "🟡", f"Las decisiones se repiten {obs_repite:.0%} (azar: {perm.mean():.0%}). Hay señal, "
        f"pero la persistencia ya logra F1 macro {f1_pers:.2f}."),
    ("3. Estabilidad en el tiempo", "🔴", f"La mezcla cambia entre períodos (chi² p = {chi2_per.pvalue:.4f}, V de Cramér = {cramer:.2f})."),
    ("4. Disponibilidad de datos (fuga)", "🟡", "Reglas de rezago aproximadas y sin versiones históricas; la fecha de publicación de la EEE debe verificarse."),
    ("5. Valor frente a la línea base", "🟡", f"Macro público: F1 macro {f1_macro_solo:.2f} (solo) y {f1_estmac:.2f} (con estado) < persistencia {f1_pers:.2f}. "
        f"EEE publicada: {f1_eee:.2f} (Δ {d_eee['delta_vs_persistencia']:+.2f})."),
    ("6. Incertidumbre de la evaluación", "🔴", f"Evaluación de {len(IDX)} reuniones ({test_c['baja']} bajas, {test_c['sube']} subidas); "
        f"el IC95% del F1 macro de la persistencia mide {anch_pers:.2f} de ancho."),
], columns=["Condición", "Semáforo", "Evidencia"])
pd.set_option("display.max_colwidth", 200)
display(tablero)

,Condición,Semáforo,Evidencia
0,1. Tamaño de muestra,🔴,161 decisiones; al inicio solo 9 bajas y 12 subidas (~0.9 variable por la regla de 10 eventos).
1,2. Dependencia temporal,🟡,"Las decisiones se repiten 76% (azar: 47%). Hay señal, pero la persistencia ya logra F1 macro 0.69."
2,3. Estabilidad en el tiempo,🔴,"La mezcla cambia entre períodos (chi² p = 0.0006, V de Cramér = 0.27)."
3,4. Disponibilidad de datos (fuga),🟡,Reglas de rezago aproximadas y sin versiones históricas; la fecha de publicación de la EEE debe verificarse.
4,5. Valor frente a la línea base,🟡,Macro público: F1 macro 0.40 (solo) y 0.60 (con estado) < persistencia 0.69. EEE publicada: 0.84 (Δ +0.15).
5,6. Incertidumbre de la evaluación,🔴,"Evaluación de 101 reuniones (22 bajas, 15 subidas); el IC95% del F1 macro de la persistencia mide 0.37 de ancho."


## 6. Conclusiones

**Sobre la distribución y su evolución**

1. El historial tiene **103 mantenciones (64,0 %), 31 bajas (19,3 %) y 27 subidas (16,8 %)**. Es un problema claramente desbalanceado.
2. La mezcla **cambia con el ciclo económico**: subidas en 2010–11 (7 y 5 de 12 reuniones) y en 2022 (7 de 8); bajas en 2014 (6), 2017 (4) y 2023–24 (4 y 7 de 8); y un 2016 con las 12 reuniones en mantención. Entre períodos la diferencia es estadísticamente significativa (chi² p = 0,0006).
3. Las decisiones **se repiten en rachas** (75,6 % de repetición frente a 47,1 % esperado al azar) y nunca hubo un giro inmediato de subir a bajar.

**Sobre por qué se necesita línea base y métricas por clase**

4. Un modelo que siempre dice "mantiene" logra 64 % de accuracy pero **0 % de recall** en bajas y subidas (F1 macro 0,26). El accuracy engaña; hay que reportar **precision, recall y F1 por clase**, más F1 macro o balanced accuracy.
5. Hay que fijar **dos líneas base**: "siempre mayoritaria" y, sobre todo, **"repetir la decisión anterior"** (accuracy 0,73; F1 macro 0,69), que es la que un modelo debe superar para aportar algo.

**Sobre si es confiable un modelo que anticipe la próxima reunión**

6. **Con indicadores macroeconómicos públicos solos, no.** IPC, Imacec, desempleo, dólar y cobre dan un F1 macro de 0,40 (≈ azar) y, combinados con el estado de la TPM, 0,60: **no superan** a repetir la decisión anterior.
7. **La única señal que supera la línea base es la expectativa de TPM de la EEE** (F1 macro 0,84; +0,15 sobre la persistencia, IC95 % [+0,03; +0,28]). Pero es el consenso de analistas, no un indicador macro independiente, su definición exacta (horizonte) debe confirmarse y el resultado depende de supuestos de publicación.
8. **La evidencia es frágil por diseño:** pocas decisiones por clase (al inicio 9 bajas y 12 subidas; en la evaluación 22 y 15), intervalos de confianza muy anchos (±0,18 en el F1 de la persistencia) y un problema que cambia de régimen. Agregar variables **no mejora y a menudo empeora** los modelos.

> **Veredicto:** es **razonable** construir el modelo como un *experimento de comparación frente a líneas base* (que es lo que pide el proyecto), pero **no es confiable** esperar de él una predicción operativa de la próxima reunión. El hallazgo más probable del proyecto es que "los indicadores macro públicos, por sí solos, no mejoran a la persistencia; las expectativas de mercado sí".

**Mensajes clave para el video**

- 103 / 31 / 27: el desbalance hace que el accuracy sea engañoso, así que usaremos línea base y métricas por clase.
- Las decisiones van en rachas: la línea base "repetir lo anterior" es exigente (F1 macro 0,69).
- Con poca data y un problema que cambia de régimen, los resultados tendrán incertidumbre alta; lo reportaremos con intervalos.

**Próximos pasos recomendados**

1. Reportar siempre accuracy, balanced accuracy, F1 macro y métricas por clase, junto a las dos líneas base y con intervalo bootstrap.
2. Definir el diseño **con y sin EEE** (la EEE como *benchmark*, no como variable más), y verificar el horizonte y la fecha de publicación de la serie.
3. Contrastar los rezagos de IPC, Imacec y desempleo con calendarios de publicación independientes y documentar las revisiones de datos.
4. Mantener pocas variables (≤ 3) y modelos simples y regularizados; evitar agregar indicadores "por si acaso".
5. Evaluar también por subperíodos (por ejemplo 2018–21 vs. 2022–25) y decidir qué hacer con la reunión extraordinaria del 16-03-2020 (marcarla o excluirla, con análisis de sensibilidad).

**Limitaciones de este análisis:** reglas de rezago aproximadas; sin versiones históricas de las series; evaluación con 101 reuniones; solo se probaron regresión logística y un árbol poco profundo con hiperparámetros fijos.